# FIP B8 - LangChain Q&A Assistant

**A beginner-friendly, educational walkthrough of a basic LangChain question-and-answer assistant.**

This notebook explains how the B8 project connects:

- a reusable **prompt template**,
- an **OpenAI chat model** through `ChatOpenAI`,
- a **string output parser**, and
- a **short conversation history**

using LangChain's LCEL pipe syntax.

The companion code lives in [`src/qa_assistant.py`](../src/qa_assistant.py), and the offline tests live in [`tests/test_offline.py`](../tests/test_offline.py).

> **Safety note:** this notebook contains no API key, prints no secret values, and its runnable cells use a fake chat model. Live OpenAI calls are optional and only happen after you configure your own key.


## 1. Project objectives

By the end of this notebook you should be able to:

1. Explain what a LangChain prompt template is and why it is useful.
2. Describe the role of `ChatOpenAI` and `StrOutputParser`.
3. Read the chain expression `prompt | ChatOpenAI | StrOutputParser`.
4. Keep a short conversation history containing the last **three exchanges**.
5. Load an API key safely from environment variables.
6. Validate user input and translate common API errors into friendly messages.
7. Run the assistant and its offline tests safely.
8. Explain the limitations of a small LLM-powered assistant.

**Out of scope:** fine-tuning, vector databases, retrieval-augmented generation, agents, streaming, and production deployment.


## 2. Setup

Run this notebook with the B8 virtual environment, from the project root:

```text
.venv\Scripts\python.exe -m notebook notebooks\B8_LangChain_QA_Assistant.ipynb
```

The first code cell adds the project root to `sys.path`, imports the assistant module, and prints only non-secret configuration. It does not create a model client.

If the imports fail, install the project dependencies first. `requirements.txt` currently lists **unpinned** package names, so pip resolves the newest compatible versions. For a reproducible environment, pin the versions actually installed (for example, from `pip freeze`).

```text
.venv\Scripts\python.exe -m pip install -r requirements.txt
```


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from langchain_core.language_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser

from src.qa_assistant import (
    DEFAULT_MODEL,
    MAX_EXCHANGES,
    SYSTEM_PROMPT,
    ConversationHistory,
    QAAssistant,
    build_prompt,
    translate_api_error,
    validate_question,
)

print("Project root:", PROJECT_ROOT)
print("Default model:", DEFAULT_MODEL)
print("History limit:", MAX_EXCHANGES, "exchanges")
print("Imports: OK (no model client created, no API call made)")


## 3. LangChain building blocks

### 3.1 Prompt templates

A **prompt template** is a reusable description of the messages sent to a chat model. Instead of building strings by hand, you declare variables and fill them at run time.

The B8 template has three parts:

| Part | Role | Example |
|---|---|---|
| System message | Sets behaviour and tone | "Answer clearly ... admit uncertainty" |
| History placeholder | Inserts previous turns | `MessagesPlaceholder("history")` |
| Human message | Carries the current question | `{question}` |

`ChatPromptTemplate` produces a **list of chat messages**, not one big string. That matches how modern chat models expect input.

The next cell formats the real template with an empty history.


In [ ]:
prompt = build_prompt()
messages = prompt.format_messages(history=[], question="What is LangChain?")

for message in messages:
    print(f"[{message.type}]")
    print(message.content)
    print()


The output contains a system message followed by the human question. When history is empty, `MessagesPlaceholder` contributes no messages.


### 3.2 `ChatOpenAI`

`ChatOpenAI` is LangChain's wrapper around OpenAI's chat models. The project builds it only after checking that `OPENAI_API_KEY` exists:

```python
llm = ChatOpenAI(
    model=model_name,          # from OPENAI_MODEL, fallback "gpt-6-luna"
    api_key=key,               # never hard-coded
    timeout=30,
    max_retries=2,
)
```

Key points:

- The model is **explicit**. `langchain-openai` still defaults to `gpt-3.5-turbo`, which OpenAI has shut down, so relying on the default would fail.
- `timeout=30` stops a request from hanging forever.
- `max_retries=2` lets the client retry transient failures.
- The key is passed to the client only; it is never printed or written into the notebook.


### 3.3 `StrOutputParser`

A chat model returns an `AIMessage` object. Most applications want plain text.

`StrOutputParser` takes the message content and returns a Python `str`. It is the final step of the chain and makes answers easy to print, save, or test.

The next cell demonstrates this with `FakeListChatModel`, which returns canned answers without any network request.


In [ ]:
parser = StrOutputParser()
fake_model = FakeListChatModel(
    responses=["LangChain helps you connect prompts, chat models and output parsers."]
)

raw_message = fake_model.invoke("Explain LangChain in one sentence.")
parsed_text = parser.invoke(raw_message)

print("Raw type:", type(raw_message).__name__)
print("Raw content:", raw_message.content)
print("Parsed type:", type(parsed_text).__name__)
print("Parsed text:", parsed_text)


## 4. The chain: `prompt | ChatOpenAI | StrOutputParser`

LangChain 1.x uses **LCEL** (LangChain Expression Language). The pipe operator connects runnable steps from left to right:

```text
question + history
       |
       v
ChatPromptTemplate  ->  ChatOpenAI  ->  StrOutputParser
       |                    |                 |
  list of messages      AIMessage         plain text
```

The same expression in code is:

```python
chain = prompt | ChatOpenAI(...) | StrOutputParser()
```

The chain is invoked with a dictionary:

```python
answer = chain.invoke({
    "history": history.messages,
    "question": cleaned_question,
})
```

To keep this notebook offline, the next cell uses `FakeListChatModel` in the middle. The shape of the chain is identical.


In [ ]:
demo_chain = (
    build_prompt()
    | FakeListChatModel(
        responses=[
            "A chain is a reusable sequence of steps that turns a question into an answer."
        ]
    )
    | StrOutputParser()
)

answer = demo_chain.invoke(
    {"history": [], "question": "What is a chain in LangChain?"}
)
print(answer)


## 5. Conversation history: the last three exchanges

A chat model is stateless: each request contains only what you send. To let a follow-up question refer to earlier turns, the application must send the recent conversation back with the new question.

B8 keeps:

- an in-memory list of `HumanMessage` and `AIMessage` objects;
- only the **last three exchanges** - at most six messages;
- no database, file, or cross-session memory.

Trimming matters because prompts have a token cost and a context limit. Older turns are dropped first.

`RunnableWithMessageHistory` is deprecated in current `langchain-core`, so the project keeps history explicitly. This makes the behaviour easy to see and test.


In [ ]:
history = ConversationHistory()
for number in range(1, 6):
    history.add_exchange(f"Question {number}", f"Answer {number}")

print("Stored messages:", len(history))
for message in history.messages:
    print(f"{message.type:>6}: {message.content}")


`Question 1` and `Question 2` have been dropped. The list starts with `Question 3`, so the next prompt stays small while still carrying recent context.


## 6. API key security

The key must never appear in source code, a notebook, a screenshot, a commit, or a chat message.

B8 uses environment variables:

1. `.env.example` contains placeholders only:

   ```text
   OPENAI_API_KEY=your_api_key_here
   OPENAI_MODEL=your_model_name_here
   ```

2. A local `.env` file holds the real key and is listed in `.gitignore`.
3. `python-dotenv` loads `.env` without overriding real environment variables.
4. Code checks whether the key exists, reports only **yes/no**, and raises `MissingAPIKeyError` before creating the client if the key is absent.

A ChatGPT subscription does **not** automatically include OpenAI API credits. API usage is billed separately.

**Data transmission:** when you use the live assistant, the system prompt, the last three exchanges, and your new question are sent over HTTPS to OpenAI's API for processing. Do not submit confidential, personal, or sensitive information unless your account, data policy, and local rules allow it. The offline cells in this notebook send nothing.

The next cell reports whether a key is configured. It never prints the key itself.


In [ ]:
from dotenv import load_dotenv
from src.qa_assistant import get_settings

load_dotenv(override=False)
api_key, model = get_settings()

print("Model:", model)
print("API key configured:", "yes" if api_key else "no")
print("Key value is never printed, logged, or stored in this notebook.")

# Remove the local reference so it is not accidentally displayed later.
del api_key


## 7. Input validation and error handling

### 7.1 Empty-question validation

`validate_question()` runs **before** any model call:

- `None` -> `ValueError`
- `""` -> `ValueError`
- whitespace only -> `ValueError`
- normal text -> trimmed text

Stopping bad input early avoids pointless API requests and unclear errors.


In [ ]:
candidates = ["  What is a prompt template?  ", "", "   \t\n", None]

for candidate in candidates:
    try:
        cleaned = validate_question(candidate)
    except ValueError as error:
        print(f"Rejected {candidate!r}: {error}")
    else:
        print(f"Accepted {candidate!r} -> {cleaned!r}")


### 7.2 Friendly API errors

`translate_api_error()` maps OpenAI exceptions to short messages that are safe to show to a user:

| Exception | Friendly meaning |
|---|---|
| `MissingAPIKeyError` | No `OPENAI_API_KEY` configured; raised before `ChatOpenAI` is created |
| `AuthenticationError` | Key invalid, revoked, or inactive (rejected by the API) |
| `PermissionDeniedError` | Key cannot use the selected model |
| `RateLimitError` | Rate limit or quota reached |
| `APITimeoutError` | Request timed out |
| `APIConnectionError` | Network/API unreachable |
| `BadRequestError` | Request rejected - check model name |
| `InternalServerError` | OpenAI server problem |
| `APIStatusError` | Other HTTP error |
| `OpenAIError` | Unexpected API error |

`MissingAPIKeyError` is a project-level exception. `create_assistant()` raises it when `OPENAI_API_KEY` is missing or empty, before any network request is attempted. `AuthenticationError` is different: it is an OpenAI API exception returned after a request is sent with a key that the service rejects.

The translated message never contains the key, request headers, or raw response bodies.

The next cell passes a harmless simulated `OpenAIError` through the translator. No network request is made.


In [ ]:
from openai import OpenAIError

simulated_error = OpenAIError("simulated failure for teaching only")
friendly_message = translate_api_error(simulated_error)
print(friendly_message)


## 8. How to run the assistant safely

### Command-line assistant

```text
cd path\to\B8
copy .env.example .env
```

Open `.env` and replace the value of `OPENAI_API_KEY` with your own key. Also replace `your_model_name_here` with a valid model name your account can use, for example:

```text
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-6-luna
```

The key line above is still a placeholder - do not use it as a real key. If you prefer the built-in default, delete the `OPENAI_MODEL` line entirely; `get_settings()` then falls back to `DEFAULT_MODEL = "gpt-6-luna"`. Leaving `your_model_name_here` unchanged will send an invalid model name and produce a `BadRequestError`.

Then run:

```text
.venv\Scripts\python.exe src\qa_assistant.py
```

Type a question, or `quit` / `exit` to stop. Empty input is rejected before an API call.

### Live use in Python

Only after the key is configured:

```python
from src.qa_assistant import create_assistant

assistant = create_assistant()
answer = assistant.ask("What is LangChain?")
print(answer)
```

The live call is **not** run in this notebook by default. Every cell here works offline.

### Offline tests

```text
.venv\Scripts\python.exe -m unittest discover -s tests -v
```

### Safety checklist

- [ ] `.env` exists locally but is not tracked by Git.
- [ ] No key appears in the notebook, README, tests, or outputs.
- [ ] The key is revoked immediately if it is ever exposed.
- [ ] API usage and spending limits are monitored in the OpenAI dashboard.


## 9. Offline testing with `FakeListChatModel` and `unittest`

`FakeListChatModel` returns pre-written responses. It exercises the same prompt and chain code without a key, a network connection, or API cost.

`tests/test_offline.py` contains **11 tests**:

| Area | What is checked |
|---|---|
| Validation | normal, `None`, empty, whitespace-only, trimming |
| History | pairs are stored; only the last three exchanges survive |
| Assistant | readable answers; empty input never reaches the model |
| Prompt content | earlier turns are present in later prompts |
| Missing key | `MissingAPIKeyError` is raised before `ChatOpenAI` is created |

The missing-key test patches `ChatOpenAI` and asserts it was **never called**. The history test uses a LangChain callback to inspect the actual messages sent to the fake model.

Expected summary: `Ran 11 tests ... OK`.


In [ ]:
offline_assistant = QAAssistant(
    llm=FakeListChatModel(
        responses=[
            "A prompt template keeps reusable instructions separate from user input.",
            "Conversation history lets the model refer back to earlier turns.",
        ]
    )
)

print(offline_assistant.ask("Why use a prompt template?"))
print(offline_assistant.ask("Why keep history?"))
print("Messages in history:", len(offline_assistant.history))


## 10. Five illustrative Q&A examples

The answers below are **illustrative**, written to match the assistant's prompt and project scope. They are **not captured API responses**; a live model may phrase them differently. Check important claims against current documentation.

| # | Question | Illustrative answer |
|---|---|---|
| 1 | What is LangChain? | LangChain is a framework for building applications around large language models. It provides reusable prompts, model wrappers, output parsers, and a way to connect those steps into chains. |
| 2 | What is a prompt template, and why use one? | A prompt template stores reusable instructions and named variables. It keeps the system message and question format consistent and makes prompts easier to test and change. |
| 3 | What does `ChatOpenAI` do? | It adapts OpenAI's chat models to LangChain's chat-model interface so the model can be used as one step in a chain. The project sets the model explicitly and reads the key from the environment. |
| 4 | What does `StrOutputParser` do? | It extracts the text from the model's `AIMessage` and returns a plain Python string, which is easier to print, save, or compare. |
| 5 | Why keep only the last three exchanges? | Recent turns provide useful context while limiting prompt size, token cost, and the risk of hitting the context limit. Older turns are dropped first. |

A live question from this notebook or from `src/qa_assistant.py` is transmitted to OpenAI's API, as described in Section 6. The offline examples in Section 9 do not send anything.


## 11. Conclusion

B8 is intentionally small, but it demonstrates the core shape of a LangChain application:

- a clear **system prompt** and reusable **prompt template**;
- an **OpenAI chat model** configured from environment variables;
- a **chain** that ends in plain text;
- **short conversation history** with explicit trimming;
- **validation** before requests;
- **friendly error handling** that never leaks secrets;
- **offline tests** that run without a key.

### Limitations to remember

- Answers can be wrong, incomplete, or outdated.
- The assistant has no document retrieval or database, so it only knows the model's training data and the current conversation.
- History is lost when the process ends.
- API calls cost money and depend on network access, account access, and model availability.
- The project is educational, not a production service: it has no authentication, rate limiting, logging policy, or monitoring.

### Good next steps

1. Add a small evaluation set with expected answer content.
2. Add a CLI flag for a different `OPENAI_MODEL`.
3. Explore LangGraph persistence if durable conversation history is needed.
4. Add retrieval only after the basic chain is understood and tested.
